# Python 数据结构工具

这一节补充标准库里常用于工程代码的数据结构工具。它们不是完整算法题解，而是帮助你更清晰地维护有序集合、优先级队列、双端队列、计数器和有序映射。


## `bisect`

`bisect` 适合维护已经排序的小到中等规模列表。它的插入成本仍然是 `O(n)`，但查找插入位置是 `O(log n)`，适合读多写少或数据量不大的场景。


In [ ]:

import bisect

scores: list[int] = []
for score in [80, 95, 70, 88, 95]:
    bisect.insort(scores, score)

print("Sorted scores:", scores)
print("First >= 88:", bisect.bisect_left(scores, 88))
print("First > 88:", bisect.bisect_right(scores, 88))


def grade_rank(sorted_scores: list[int], target: int) -> int:
    """计算分数在有序列表中的排名位置。

    Args:
        sorted_scores: 从小到大排序的分数列表。
        target: 目标分数。

    Returns:
        目标分数可插入的位置。
    """
    return bisect.bisect_left(sorted_scores, target)


print("Rank index:", grade_rank(scores, 90))


## `heapq`

`heapq` 默认是最小堆，适合实现 Top K、优先级任务调度和流式最值维护。需要最大堆时常见做法是存入负数或反向排序键。


In [ ]:

import heapq
from dataclasses import dataclass, field


@dataclass(order=True)
class Task:
    """带优先级的任务。

    Attributes:
        priority: 优先级，数值越小越先执行。
        name: 任务名称。
    """

    priority: int
    name: str = field(compare=False)


tasks = [Task(3, "send-email"), Task(1, "sync-user"), Task(2, "build-report")]
heapq.heapify(tasks)

while tasks:
    print("Run:", heapq.heappop(tasks))

numbers = [9, 4, 7, 1, 3, 8]
print("Top 3:", heapq.nlargest(3, numbers))
print("Smallest 2:", heapq.nsmallest(2, numbers))


## `deque`

`collections.deque` 是双端队列，头尾插入和弹出都是 `O(1)`。它适合固定长度窗口、生产消费队列和 BFS 队列。


In [ ]:

from collections import deque

window: deque[int] = deque(maxlen=3)
for value in [10, 20, 30, 40]:
    window.append(value)
    print("Window:", list(window), "Sum:", sum(window))

queue: deque[str] = deque(["root"])
while queue:
    node = queue.popleft()
    print("Visit:", node)
    if node == "root":
        queue.extend(["left", "right"])


## `Counter` 和 `defaultdict`

`Counter` 适合频次统计，`defaultdict` 适合把“如果不存在就初始化”的样板代码收敛掉。


In [ ]:

from collections import Counter, defaultdict

words = ["ray", "dask", "ray", "celery", "arq", "celery", "ray"]
counts = Counter(words)
print("Most common:", counts.most_common(2))

grouped: defaultdict[str, list[str]] = defaultdict(list)
for tool in words:
    grouped[tool[0]].append(tool)

print("Grouped:", dict(grouped))


## `OrderedDict`

Python 3.7+ 的普通 `dict` 已经保留插入顺序，但 `OrderedDict` 仍然适合需要移动顺序、实现 LRU 或明确表达“顺序是业务语义”的场景。


In [ ]:

from collections import OrderedDict

cache: OrderedDict[str, int] = OrderedDict()
cache["a"] = 1
cache["b"] = 100
cache["c"] = 3

cache.move_to_end("a")
cache["d"] = -1
oldest_key, oldest_value = cache.popitem(last=False)

print("Cache:", cache)
print("Evicted:", oldest_key, oldest_value)
